# FunctorFlow PTB KET Language Model

This notebook is the first real-data FunctorFlow KET path in the repo. It uses
the local Penn Treebank corpus, instantiates a FunctorFlow-backed KET language
model, runs a short training loop, and evaluates validation perplexity.


In [ ]:
from pathlib import Path
import sys

def _bootstrap_functorflow() -> Path:
    cwd = Path.cwd().resolve()
    candidates = [cwd, *cwd.parents]
    for candidate in candidates:
        package_root = candidate / "FunctorFlow"
        if (package_root / "__init__.py").exists():
            if str(candidate) not in sys.path:
                sys.path.insert(0, str(candidate))
            return candidate
    raise RuntimeError(
        "Could not find the FunctorFlow package. Run this notebook from the repo "
        "root or from FunctorFlow/notebooks."
    )

REPO_ROOT = _bootstrap_functorflow()
print(f"FunctorFlow repo root: {REPO_ROOT}")


In [ ]:
from FunctorFlow.ket_lm import (
    FunctorFlowKETLanguageModel,
    KETLanguageModelConfig,
    estimate_perplexity,
    load_word_language_modeling_corpus,
    pick_device,
    train_language_model,
)

device = pick_device("cuda")
corpus = load_word_language_modeling_corpus("ptb")
config = KETLanguageModelConfig.historical_ptb_smoke()
corpus.name, corpus.vocab_size, len(corpus.train_ids), len(corpus.valid_ids), config


## Build The Model

We keep the first pass intentionally small so the notebook stays runnable on a
single workstation while still exercising the real FunctorFlow KET head.


In [ ]:
model = FunctorFlowKETLanguageModel(
    corpus.vocab_size,
    config=config,
)
model


## Train Briefly On PTB

This is a smoke-scale run for tutorial development. The next pass can lengthen
training, swap to `KETLanguageModelConfig.historical_ptb_reference()`, or move
the same model family over to Wiki-2 once the FunctorFlow execution path is
stable.


In [ ]:
history = train_language_model(
    model,
    corpus,
    steps=50,
    block_size=128,
    batch_size=16,
    lr=2e-3,
    device=device,
)
history


In [ ]:
valid_ppl = estimate_perplexity(
    model,
    corpus.valid_ids,
    block_size=128,
    batch_size=16,
    device=device,
)
valid_ppl


## Next Step

Once this PTB path looks healthy, the immediate follow-on is to add a matching
Wiki-2 notebook and then start swapping in one of the larger historical KET
language-model variants already present elsewhere in the repo.
